In [1]:
#import drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
#import the necessary libraries
import numpy as np
import pandas as pd

In [3]:
#load the dataset
df = pd.read_csv('/content/drive/MyDrive/DS Projects/Logistics operation/logistics_operations_sample_data.csv')

In [4]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 13 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   order_id                 100 non-null    object 
 1   order_date               100 non-null    object 
 2   region                   100 non-null    object 
 3   warehouse_id             100 non-null    object 
 4   carrier                  100 non-null    object 
 5   product_category         100 non-null    object 
 6   quantity                 100 non-null    int64  
 7   order_value_eur          100 non-null    float64
 8   freight_cost_eur         100 non-null    float64
 9   estimated_delivery_date  100 non-null    object 
 10  actual_delivery_date     74 non-null     object 
 11  delivery_status          100 non-null    object 
 12  customer_rating          74 non-null     float64
dtypes: float64(3), int64(1), object(9)
memory usage: 10.3+ KB


In [5]:
df.head()

,order_id,order_date,region,warehouse_id,carrier,product_category,quantity,order_value_eur,freight_cost_eur,estimated_delivery_date,actual_delivery_date,delivery_status,customer_rating
0,ORD-0001,2026-01-27,Central,WH-01,UPS,Furniture,1,1464.04,65.65,2026-02-02,2026-02-03,Delivered,4.0
1,ORD-0002,2026-08-24,East,WH-02,DHL,Home & Kitchen,8,1238.57,39.92,2026-08-31,2026-09-01,Delivered,4.0
2,ORD-0003,2026-05-16,North,WH-04,FedEx,Furniture,7,1143.18,32.72,2026-05-19,2026-05-23,Delivered,1.0
3,ORD-0004,2026-10-19,West,WH-01,DPD,Clothing,5,252.58,59.33,2026-10-25,NaN,Delayed,NaN
4,ORD-0005,2026-01-14,East,WH-02,FedEx,Furniture,1,304.47,14.52,2026-01-18,2026-01-22,Delivered,3.0


In [6]:
df.describe()

,quantity,order_value_eur,freight_cost_eur,customer_rating
count,100.000000,100.000000,100.000000,74.000000
mean,5.270000,817.759400,40.265900,3.094595
std,2.751877,427.904952,23.584736,1.472756
min,1.000000,31.640000,4.600000,1.000000
25%,3.000000,489.167500,19.065000,2.000000
50%,5.000000,861.095000,37.615000,3.000000
75%,7.000000,1186.015000,61.630000,4.000000
max,10.000000,1495.630000,83.380000,5.000000


Check Missing values

In [7]:
#check for missing values
missing_values = df.isnull().sum()
missing_percent = (missing_values / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing Values': missing_values,
    'Percentage (%)': missing_percent.round(2)
})

# Filter only columns with missing data
missing_df = missing_df[missing_df['Missing Values'] > 0]

if missing_df.empty:
    print("No missing values in the dataset.")
else:
    print(missing_df.sort_values(by='Percentage (%)', ascending=False))

                      Missing Values  Percentage (%)
actual_delivery_date              26            26.0
customer_rating                   26            26.0


Print Unique values in the columns

In [8]:
for col in df.select_dtypes(include=['object', 'category']).columns:
    print(f"\nUnique value counts for '{col}':")
    print(df[col].value_counts())


Unique value counts for 'order_id':
order_id
ORD-0001    1
ORD-0002    1
ORD-0003    1
ORD-0004    1
ORD-0005    1
           ..
ORD-0096    1
ORD-0097    1
ORD-0098    1
ORD-0099    1
ORD-0100    1
Name: count, Length: 100, dtype: int64

Unique value counts for 'order_date':
order_date
2026-01-26    3
2026-07-10    3
2026-03-15    3
2026-05-23    3
2026-05-18    2
             ..
2026-06-18    1
2026-09-14    1
2026-05-22    1
2026-01-11    1
2026-01-20    1
Name: count, Length: 82, dtype: int64

Unique value counts for 'region':
region
Central    27
North      24
West       17
South      17
East       15
Name: count, dtype: int64

Unique value counts for 'warehouse_id':
warehouse_id
WH-02    27
WH-04    26
WH-03    25
WH-01    22
Name: count, dtype: int64

Unique value counts for 'carrier':
carrier
UPS      32
DHL      25
FedEx    24
DPD      19
Name: count, dtype: int64

Unique value counts for 'product_category':
product_category
Food              26
Home & Kitchen    21
Furniture

In [9]:
# Check duplicate order IDs
print(df["order_id"].duplicated().sum())

0


In [10]:
# Remove duplicate order IDs if they exist
df = df.drop_duplicates(subset="order_id")

In [11]:
date_columns = [
    "order_date",
    "estimated_delivery_date",
    "actual_delivery_date"
]

for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors="coerce")

In [12]:
# Delivery duration
df["delivery_days"] = (
    df["actual_delivery_date"] - df["order_date"]
).dt.days

In [13]:
# Delivery delay
df["delay_days"] = (
    df["actual_delivery_date"] -
    df["estimated_delivery_date"]
).dt.days

In [14]:
# Delivery performance classification
df["delivery_performance"] = np.select(
    [
        df["delivery_status"].eq("Delivered") &
        df["delay_days"].le(0),

        df["delivery_status"].eq("Delivered") &
        df["delay_days"].gt(0)
    ],
    ["On Time", "Late"],
    default="Not Delivered"
)

In [24]:
# Ensure delivery_performance is defined, then display the head
df["delivery_performance"] = np.select(
    [
        df["delivery_status"].eq("Delivered") &
        df["delay_days"].le(0),

        df["delivery_status"].eq("Delivered") &
        df["delay_days"].gt(0)
    ],
    ["On Time", "Late"],
    default="Not Delivered"
)

print(df[[
    "order_id",
    "delivery_days",
    "delay_days",
    "delivery_performance"
]].head(10))

   order_id  delivery_days  delay_days delivery_performance
0  ORD-0001            7.0         1.0                 Late
1  ORD-0002            8.0         1.0                 Late
2  ORD-0003            7.0         4.0                 Late
3  ORD-0004            NaN         NaN        Not Delivered
4  ORD-0005            8.0         4.0                 Late
5  ORD-0006            5.0         1.0                 Late
6  ORD-0007            NaN         NaN        Not Delivered
7  ORD-0008            NaN         NaN        Not Delivered
8  ORD-0009            6.0         2.0                 Late
9  ORD-0010            7.0         2.0                 Late


In [17]:
delivered = df[df["delivery_status"] == "Delivered"]

total_orders = df["order_id"].nunique()

delivered_orders = delivered["order_id"].nunique()

In [18]:
on_time_orders = (
    delivered["delivery_performance"] == "On Time"
).sum()

on_time_rate = (
    on_time_orders / delivered_orders * 100
    if delivered_orders else 0
)

In [19]:
average_delivery_days = delivered["delivery_days"].mean()

total_freight_cost = df["freight_cost_eur"].sum()

print("Total Orders:", total_orders)
print("Delivered Orders:", delivered_orders)
print("On-Time Delivery Rate:", round(on_time_rate, 2), "%")
print("Average Delivery Days:", round(average_delivery_days, 2))
print("Total Freight Cost:", round(total_freight_cost, 2))

Total Orders: 100
Delivered Orders: 74
On-Time Delivery Rate: 39.19 %
Average Delivery Days: 5.8
Total Freight Cost: 4026.59


Export the prepared dataset:

In [26]:
df.to_csv(
    "/content/drive/MyDrive/DS Projects/Logistics operation/logistics_cleaned.csv",
    index=False
)